# Exploratory Data Analysis — MovieLens 1M

Run `python train.py` from the project root at least once before using this notebook, or run the cells below directly — they use the same `src/` preprocessing pipeline as `train.py`, so results are always generated from the real dataset.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import run_preprocessing_pipeline

sns.set_theme(style="whitegrid")
movies, ratings, users, stats = run_preprocessing_pipeline()
stats.to_dict()

## Rating distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=ratings["rating"])
plt.title("Rating Distribution")
plt.show()

## Ratings per movie / per user (long-tail check)

In [ ]:
per_movie = ratings.groupby("movieId").size()
per_user = ratings.groupby("userId").size()
print("Ratings per movie — median:", per_movie.median(), "max:", per_movie.max())
print("Ratings per user — median:", per_user.median(), "max:", per_user.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(per_movie, bins=50, log_scale=(False, True), ax=axes[0]).set_title("Ratings per Movie")
sns.histplot(per_user, bins=50, log_scale=(False, True), ax=axes[1]).set_title("Ratings per User")
plt.tight_layout()
plt.show()

## Most common genres

In [ ]:
genre_counts = movies["genres_clean"].str.split().explode().value_counts()
plt.figure(figsize=(7, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, orient="h")
plt.title("Movie Count by Genre")
plt.tight_layout()
plt.show()

## Top-rated movies (min. 20 ratings)

In [ ]:
movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"])
movie_stats = movie_stats[movie_stats["count"] >= 20].sort_values("mean", ascending=False).head(15)
movie_stats.join(movies.set_index("movieId")[["title"]])